# BuffettRAG: LoRA fine-tune Qwen2.5-7B-Instruct (FT-r3) on ft_v6, prompt v2, long served context
Rows carry the exact serving chat (system + user as llama_provider.split_prompt builds them for PROMPT_VERSION=v2) with 8, 12 or 16 served passages; loss on the assistant answer only. MAX_SEQ is derived from the data (p99 rounded up to 512, cap 12288); longer rows are dropped and counted. LoRA rank/alpha/target modules/learning rate and the effective batch are FT-r1's. One adapter per epoch, valid loss each epoch, then one merged q4_k_m GGUF of the final model. Re-run top to bottom after a dropped session: training resumes from the last checkpoint, a finished export is skipped.


## a. Settings

In [ ]:
DATA_NAME = 'ft_v6'
DRIVE_DIR = '/content/drive/MyDrive/buffettrag'
BUNDLE = DRIVE_DIR + '/colab-bundle-7b.zip'
BUNDLE_DIR = '/content/bundle'
OUT_DIR = DRIVE_DIR + '/run-7b-r3'
BASE_MODEL = 'unsloth/Qwen2.5-7B-Instruct-bnb-4bit'
MAX_SEQ_OVERRIDE = None  # None: p99 of the data rounded up to the next 512
MAX_SEQ_CAP = 12288      # rows longer than the cap are never trained on
# LoRA / optimiser: identical to FT-r1 (colab/train_7b_lora.ipynb at 7c522f5, run-7b-r1): r=16, alpha=32, lr=1e-4, all 7 projections,
# per-device batch 1 x grad-accum 2 = effective batch 2 (long rows: batch 1 is the only one that fits a 16 GB T4)
EPOCHS = 2
LR = 1e-4
RANK = 16
ALPHA = 32
BATCH = 1
GRAD_ACCUM = 2
SEED = 13
EXPORT_BASE_GGUF = False


## b. GPU check

In [ ]:
import subprocess, torch
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
if not torch.cuda.is_available():
    raise SystemExit('No GPU. Runtime > Change runtime type > GPU, then re-run.')
GPU_NAME = torch.cuda.get_device_name(0)
print('GPU:', GPU_NAME)

## c. Mount Drive, unzip bundle, verify SHA256SUMS

In [ ]:
import hashlib, os, zipfile
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(OUT_DIR, exist_ok=True)
assert os.path.isfile(BUNDLE), 'bundle not found: ' + BUNDLE
with zipfile.ZipFile(BUNDLE) as zf:
    zf.extractall(BUNDLE_DIR)
bad = 0
for line in open(BUNDLE_DIR + '/SHA256SUMS'):
    if not line.strip():
        continue
    want, rel = line.strip().split(None, 1)
    got = hashlib.sha256(open(BUNDLE_DIR + '/' + rel.strip(), 'rb').read()).hexdigest()
    ok = got == want
    bad += not ok
    print('OK ' if ok else 'BAD', rel)
if bad:
    raise SystemExit(f'{bad} checksum mismatch(es); stopping.')


## d. Install unsloth

In [ ]:
!pip install -q unsloth
import torch, transformers, trl, unsloth
print('torch', torch.__version__, '| transformers', transformers.__version__, '| trl', trl.__version__, '| unsloth', getattr(unsloth, '__version__', '?'))

## e. Load base model + LoRA (max length = the cap; the real MAX_SEQ is set from the data in the next step)

In [ ]:
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(BASE_MODEL, max_seq_length=MAX_SEQ_CAP, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=RANK, lora_alpha=ALPHA, lora_dropout=0, bias='none',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    use_gradient_checkpointing='unsloth', random_state=SEED)


## f. Data: serving chat template, MAX_SEQ from the data, drop longer rows (assistant-only loss is applied in the trainer cell)

In [ ]:
import json, math
from datasets import Dataset

MANIFEST = json.load(open(f'{BUNDLE_DIR}/{DATA_NAME}/manifest.json'))
assert MANIFEST.get('prompt_version') == 'v2', MANIFEST.get('prompt_version')
print('manifest: prompt_version', MANIFEST['prompt_version'], '| passages_mix', MANIFEST.get('passages_mix'), '| k_histogram', MANIFEST.get('k_histogram'))

def load(path):
    return [json.loads(l)['messages'] for l in open(path) if l.strip()]

def encode(rows):
    # ChatML via the tokenizer's own template, exactly what llama.cpp applies at serving: system, user, then the assistant answer
    out = []
    for msgs in rows:
        assert [m['role'] for m in msgs] == ['system', 'user', 'assistant'], [m['role'] for m in msgs]
        text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
        out.append((text, len(tokenizer(text, add_special_tokens=False)['input_ids'])))
    return out

train_rows = load(f'{BUNDLE_DIR}/{DATA_NAME}/train.jsonl')
valid_rows = load(f'{BUNDLE_DIR}/{DATA_NAME}/valid.jsonl')
systems = {m[0]['content'] for m in train_rows + valid_rows}
assert len(systems) == 1, f'{len(systems)} different system prompts'
print('system prompt (one for all rows):', next(iter(systems))[:90].replace(chr(10), ' '), '...')
assert all(m[1]['content'].lstrip().startswith(('BEGIN UNTRUSTED PASSAGES', 'BEGIN RECENT CONVERSATION')) for m in train_rows + valid_rows)

train_enc, valid_enc = encode(train_rows), encode(valid_rows)
lengths = sorted(n for _, n in train_enc + valid_enc)
P50, P95, P99 = (lengths[min(len(lengths) - 1, math.ceil(q / 100 * len(lengths)) - 1)] for q in (50, 95, 99))  # nearest rank
MAX_SEQ = MAX_SEQ_OVERRIDE or min(MAX_SEQ_CAP, math.ceil(P99 / 512) * 512)
print(f'tokens prompt+answer: p50={P50} p95={P95} p99={P99} max={lengths[-1]} -> MAX_SEQ={MAX_SEQ} (cap {MAX_SEQ_CAP})')

def keep(enc, name):
    kept = [{'text': t} for t, n in enc if n <= MAX_SEQ]
    dropped = len(enc) - len(kept)
    print(f'{name}: rows={len(enc)} kept={len(kept)} dropped(>{MAX_SEQ})={dropped}')
    return Dataset.from_list(kept), dropped

train_ds, TRAIN_DROPPED = keep(train_enc, 'train')
valid_ds, VALID_DROPPED = keep(valid_enc, 'valid')


## g. Train (resumes from checkpoint if present)

In [ ]:
import glob, time
from trl import SFTTrainer, SFTConfig
from unsloth import is_bf16_supported
from unsloth.chat_templates import train_on_responses_only

CKPT_DIR = OUT_DIR + '/checkpoints'
args = SFTConfig(
    output_dir=CKPT_DIR, dataset_text_field='text', max_seq_length=MAX_SEQ, packing=False,
    num_train_epochs=EPOCHS, learning_rate=LR, per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACCUM, eval_strategy='epoch',
    per_device_eval_batch_size=1, logging_steps=5, save_strategy='epoch',
    save_total_limit=max(2, EPOCHS), fp16=not is_bf16_supported(), bf16=is_bf16_supported(), seed=SEED,
    optim='adamw_8bit', lr_scheduler_type='linear', warmup_steps=5, weight_decay=0.01, report_to='none')
trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=train_ds, eval_dataset=valid_ds, args=args)
trainer = train_on_responses_only(trainer, instruction_part='<|im_start|>user\n', response_part='<|im_start|>assistant\n')
# loss on the assistant answer only: what is left unmasked must be exactly the answer (+ <|im_end|>)
for i in range(3):
    ex = trainer.train_dataset[i]
    live = tokenizer.decode([t for t in ex['labels'] if t != -100]).strip()
    answer = train_rows[i][2]['content'].strip()
    assert live == answer + '<|im_end|>' or live == answer, (live[:120], answer[:120])
print('assistant-only loss check OK on 3 rows')
resume = bool(glob.glob(CKPT_DIR + '/checkpoint-*'))
print('resume from checkpoint:', resume)
torch.cuda.reset_peak_memory_stats()
t0 = time.time()
result = trainer.train(resume_from_checkpoint=True) if resume else trainer.train()
SECONDS = round(time.time() - t0, 1)
PEAK_GPU_GB = round(torch.cuda.max_memory_allocated() / 2**30, 2)
print(result)
print('peak GPU memory allocated (GB):', PEAK_GPU_GB, 'of', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))


## h. Save final adapter, one adapter per epoch, metrics

In [ ]:
import shutil
model.save_pretrained(OUT_DIR + '/adapter')
tokenizer.save_pretrained(OUT_DIR + '/adapter')

# One adapter per epoch: the epoch-end checkpoints are kept (never deleted); copy their adapter files next to the final adapter.
EPOCH_ADAPTERS = {}
for d in sorted(glob.glob(CKPT_DIR + '/checkpoint-*')):
    ep = round(json.load(open(d + '/trainer_state.json'))['epoch'])
    assert os.path.isfile(d + '/adapter_model.safetensors'), 'no adapter in ' + d
    dst = OUT_DIR + f'/adapter_e{ep}'
    shutil.copytree(d, dst, dirs_exist_ok=True, ignore=shutil.ignore_patterns('optimizer.pt', 'scheduler.pt', 'rng_state*', 'scaler.pt'))
    EPOCH_ADAPTERS[ep] = dst
assert sorted(EPOCH_ADAPTERS) == list(range(1, EPOCHS + 1)), f'epoch adapters found: {sorted(EPOCH_ADAPTERS)}'
print('epoch adapters:', EPOCH_ADAPTERS)

hist = trainer.state.log_history
train_losses = [h['loss'] for h in hist if 'loss' in h]
metrics = {
    'gpu': GPU_NAME, 'torch': torch.__version__, 'unsloth': getattr(unsloth, '__version__', '?'),
    'data_name': DATA_NAME, 'max_seq': MAX_SEQ, 'max_seq_cap': MAX_SEQ_CAP,
    'token_p50_p95_p99_max': [P50, P95, P99, lengths[-1]], 'peak_gpu_gb': PEAK_GPU_GB,
    'lora': {'rank': RANK, 'alpha': ALPHA, 'lr': LR, 'batch': BATCH, 'grad_accum': GRAD_ACCUM}, 'manifest_k_histogram': MANIFEST.get('k_histogram'),
    'epochs': EPOCHS, 'steps': trainer.state.global_step, 'train_rows': len(train_ds),
    'dropped_rows': {'train': TRAIN_DROPPED, 'valid': VALID_DROPPED},
    'final_train_loss': result.training_loss if result.training_loss is not None else (train_losses[-1] if train_losses else None),
    'last_logged_train_loss': train_losses[-1] if train_losses else None,
    'eval_loss_per_epoch': [{'epoch': round(h['epoch']), 'step': h['step'], 'eval_loss': h['eval_loss']} for h in hist if 'eval_loss' in h],
    'eval_loss_history': [{'step': h['step'], 'eval_loss': h['eval_loss']} for h in hist if 'eval_loss' in h],
    'seconds': SECONDS, 'seconds_note': 'this session only (a resumed run counts the resumed part)',
}
json.dump(metrics, open(OUT_DIR + '/metrics.json', 'w'), indent=2)
print(json.dumps(metrics, indent=2))

## i. Merge the final adapter + export one q4_k_m GGUF

In [ ]:
import glob, shutil, gc
FT_GGUF = OUT_DIR + '/buffett-qwen2.5-7b-ft-r3-q4_k_m.gguf'

_sha_cache = {}
def sha256_file(p):
    key = (p, os.path.getsize(p))
    if key not in _sha_cache:
        h = hashlib.sha256()
        with open(p, 'rb') as f:
            for b in iter(lambda: f.read(1 << 24), b''):
                h.update(b)
        _sha_cache[key] = h.hexdigest()
    return _sha_cache[key]

def write_sums():
    with open(OUT_DIR + '/SHA256SUMS', 'w') as f:
        for p in sorted(glob.glob(OUT_DIR + '/*.gguf')):
            f.write(f'{sha256_file(p)}  {os.path.basename(p)}\n')

def export_gguf(m, tok, work, dest):
    # unsloth merges the LoRA into a temp dir relative to the cwd, so work in /tmp and clean up after copying
    shutil.rmtree(work, ignore_errors=True)
    os.makedirs(work)
    cwd = os.getcwd()
    os.chdir(work)
    try:
        m.save_pretrained_gguf(work + '/out', tok, quantization_method='q4_k_m')
        cands = [p for p in glob.glob(work + '*/**/*.gguf', recursive=True) if 'q4_k_m' in os.path.basename(p).lower()]
        assert len(cands) == 1, f'expected one q4_k_m gguf, found {cands}'
        shutil.copy(cands[0], dest)
    finally:
        os.chdir(cwd)
        shutil.rmtree(work, ignore_errors=True)
        for p in glob.glob(work + '*'):
            shutil.rmtree(p, ignore_errors=True)

# the training model is no longer needed; the final adapter is reloaded on its own and merged by the export
globals().pop('trainer', None); globals().pop('model', None)
gc.collect(); torch.cuda.empty_cache()

if os.path.exists(FT_GGUF):
    print('exists, skipping:', FT_GGUF)
else:
    m, tok = FastLanguageModel.from_pretrained(OUT_DIR + '/adapter', max_seq_length=MAX_SEQ, load_in_4bit=True)
    try:
        export_gguf(m, tok, '/tmp/gguf_ft', FT_GGUF)
    finally:
        del m, tok
        gc.collect(); torch.cuda.empty_cache()
print(FT_GGUF, os.path.getsize(FT_GGUF), 'bytes\nsha256', sha256_file(FT_GGUF))
write_sums()


## j. Export untouched base GGUF (same tool chain, off by default)

In [ ]:
BASE_GGUF = OUT_DIR + '/qwen2.5-7b-instruct-q4_k_m.gguf'
if not EXPORT_BASE_GGUF:
    print('EXPORT_BASE_GGUF is False, skipping')
elif os.path.exists(BASE_GGUF):
    print('exists, skipping:', BASE_GGUF)
else:
    base_model, base_tok = FastLanguageModel.from_pretrained(BASE_MODEL, max_seq_length=MAX_SEQ, load_in_4bit=True)
    export_gguf(base_model, base_tok, '/tmp/gguf_base', BASE_GGUF)
    del base_model, base_tok
    gc.collect(); torch.cuda.empty_cache()
if os.path.exists(BASE_GGUF):
    print(BASE_GGUF, os.path.getsize(BASE_GGUF), 'bytes\nsha256', sha256_file(BASE_GGUF))
write_sums()
print(open(OUT_DIR + '/SHA256SUMS').read())

## k. Done

In [ ]:
print('ALL_DONE')
for root, _, files in os.walk(OUT_DIR):
    if '/checkpoints' in root:
        continue
    for f in sorted(files):
        p = os.path.join(root, f)
        print(f'{os.path.getsize(p):>14,}  {p}')